## Ingesting the documents

In [1]:
from langchain_community.document_loaders.csv_loader import CSVLoader

vrmuseum_docs = CSVLoader(
    file_path='../../data/processed/vrmuseum.csv',
    content_columns=['content'],
    metadata_columns=['title', 'specimen_name'],
    encoding='utf-8'
).load()

In [3]:
import os
import chromadb
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma

model_name = 'text-embedding-3-large'
embeddings = AzureOpenAIEmbeddings(model=model_name, dimensions=1024)

chroma_client = chromadb.CloudClient(
    tenant=os.getenv('VRMUSEUM_CHROMA_TENANT'),
    database=os.getenv('VRMUSEUM_CHROMA_DATABASE'),
    api_key=os.getenv('VRMUSEUM_CHROMA_API_KEY')
)

vector_store = Chroma(
    collection_name='main', 
    embedding_function=embeddings,
    client=chroma_client
)

In [7]:
from uuid import uuid4

index_documents = True

def split_list(input_list, chunk_size=5461):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

if index_documents:
    for batch_docs in split_list(vrmuseum_docs):
        batch_ids = [str(uuid4()) for _ in range(len(vrmuseum_docs))]
        vector_store.add_documents(documents=batch_docs, ids=batch_ids)

In [8]:
dense_retriever = vector_store.as_retriever(search_kwargs={'k': 5})

In [9]:
# Test retrieval of five results
results = dense_retriever.invoke('preservation')

for doc in results:
    print(doc.metadata['specimen_name'])

RM 2725-01 croc mummy
RM 2725-02 croc mummy
Fossil ammonite (McKee collection)
RM 2727.2 ibis mummy fake
RM 2728.1 cat mummy
